<a href="https://colab.research.google.com/github/hgshreyas/BIS_LAB/blob/main/1BM24CS108_Traffic_Signal_GWO.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
import random

# ---------------- USER INPUT ----------------

n = int(input("Enter number of traffic signals: "))

arrival = list(map(int, input("Enter vehicle arrival rates: ").split()))
queue = list(map(int, input("Enter queue lengths: ").split()))
waiting = list(map(int, input("Enter average waiting times: ").split()))

# ---------------- PARAMETERS ----------------

N = 10
CS_ITERATIONS = 30
GWO_ITERATIONS = 100
RUNS = 10

LOWER = 10
UPPER = 60
PA = 0.25

# ---------------- FITNESS FUNCTION ----------------

def fitness(solution):
    cost = 0

    for i in range(n):
        demand = arrival[i] * waiting[i] + queue[i]

        cost += demand / solution[i]
        cost += 0.5 * solution[i]

    return cost


# ==================================================
# CUCKOO SEARCH (CS)
# ==================================================

def cuckoo_search():

    nests = [
        [random.uniform(LOWER, UPPER) for _ in range(n)]
        for _ in range(N)
    ]

    best = min(nests, key=fitness)[:]

    for iteration in range(CS_ITERATIONS):

        for i in range(N):

            new_nest = nests[i][:]

            # Levy flight
            for j in range(n):
                step = random.gauss(0, 1) / (
                    abs(random.gauss(0, 1)) ** (1 / 1.5) + 0.01
                )

                new_nest[j] += 2 * step
                new_nest[j] = max(LOWER, min(UPPER, new_nest[j]))

            k = random.randint(0, N - 1)

            if fitness(new_nest) < fitness(nests[k]):
                nests[k] = new_nest

        # Abandon poor nests
        nests.sort(key=fitness)

        for i in range(int(PA * N)):
            nests[-(i + 1)] = [
                random.uniform(LOWER, UPPER) for _ in range(n)
            ]

        current = min(nests, key=fitness)

        if fitness(current) < fitness(best):
            best = current[:]

    return best, fitness(best)


# ==================================================
# GREY WOLF OPTIMIZER (GWO)
# ==================================================

def grey_wolf_optimizer():

    wolves = [
        [random.uniform(LOWER, UPPER) for _ in range(n)]
        for _ in range(N)
    ]

    wolves.sort(key=fitness)

    alpha = wolves[0][:]
    beta = wolves[1][:]
    delta = wolves[2][:]

    for t in range(GWO_ITERATIONS):

        a = 2 - (2 * t / GWO_ITERATIONS)

        for i in range(N):

            current = wolves[i][:]

            for j in range(n):

                # Alpha wolf
                r1 = random.random()
                r2 = random.random()

                A1 = 2 * a * r1 - a
                C1 = 2 * r2

                D_alpha = abs(C1 * alpha[j] - current[j])
                X1 = alpha[j] - A1 * D_alpha

                # Beta wolf
                r1 = random.random()
                r2 = random.random()

                A2 = 2 * a * r1 - a
                C2 = 2 * r2

                D_beta = abs(C2 * beta[j] - current[j])
                X2 = beta[j] - A2 * D_beta

                # Delta wolf
                r1 = random.random()
                r2 = random.random()

                A3 = 2 * a * r1 - a
                C3 = 2 * r2

                D_delta = abs(C3 * delta[j] - current[j])
                X3 = delta[j] - A3 * D_delta

                # Update wolf position
                current[j] = (X1 + X2 + X3) / 3

                current[j] = max(
                    LOWER, min(UPPER, current[j])
                )

            wolves[i] = current

        # Preserve the best solution found
        wolves.sort(key=fitness)

        if fitness(wolves[0]) < fitness(alpha):
            alpha = wolves[0][:]

        # Update alpha, beta and delta
        leaders = sorted(wolves + [alpha], key=fitness)

        alpha = leaders[0][:]
        beta = leaders[1][:]
        delta = leaders[2][:]

    return alpha, fitness(alpha)


# ==================================================
# RUN BOTH ALGORITHMS
# ==================================================

cs_results = []
gwo_results = []

best_cs = None
best_gwo = None

for run in range(RUNS):

    cs_solution, cs_cost = cuckoo_search()
    gwo_solution, gwo_cost = grey_wolf_optimizer()

    cs_results.append(cs_cost)
    gwo_results.append(gwo_cost)

    if best_cs is None or cs_cost < fitness(best_cs):
        best_cs = cs_solution[:]

    if best_gwo is None or gwo_cost < fitness(best_gwo):
        best_gwo = gwo_solution[:]


# ==================================================
# COST CALCULATIONS
# ==================================================

cs_min = min(cs_results)
gwo_min = min(gwo_results)

cs_avg = sum(cs_results) / RUNS
gwo_avg = sum(gwo_results) / RUNS

# ==================================================
# OUTPUT
# ==================================================

print("\n========== CUCKOO SEARCH ==========")

print("Optimal Green Timings:",
      [round(x, 2) for x in best_cs])

print("Minimum Traffic Cost:", round(cs_min, 2))
print("Best Cost:", round(fitness(best_cs), 2))
print("Average Cost:", round(cs_avg, 2))


print("\n========== GREY WOLF OPTIMIZER ==========")

print("Optimal Green Timings:",
      [round(x, 2) for x in best_gwo])

print("Minimum Traffic Cost:", round(gwo_min, 2))
print("Best Cost:", round(fitness(best_gwo), 2))
print("Average Cost:", round(gwo_avg, 2))


print("\n========== COST COMPARISON ==========")

print("Minimum Cost Difference:",
      round(cs_min - gwo_min, 2))

print("Average Cost Difference:",
      round(cs_avg - gwo_avg, 2))

if gwo_avg < cs_avg:
    print("GWO performs better on average.")
elif cs_avg < gwo_avg:
    print("CS performs better on average.")
else:
    print("Both algorithms have equal average cost.")


Enter number of traffic signals: 3
Enter vehicle arrival rates: 20 30 25
Enter queue lengths: 10 15 12
Enter average waiting times: 40 50 35

========== CUCKOO SEARCH ==========
Optimal Green Timings: [40.02, 55.14, 42.01]
Minimum Traffic Cost: 137.41
Best Cost: 137.41
Average Cost: 137.42

========== GREY WOLF OPTIMIZER ==========
Optimal Green Timings: [40.18, 55.04, 42.04]
Minimum Traffic Cost: 137.41
Best Cost: 137.41
Average Cost: 137.41

========== COST COMPARISON ==========
Minimum Cost Difference: 0.0
Average Cost Difference: 0.01
GWO performs better on average.
